In [ ]:
%pip install genomic-benchmarks

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from genomic_benchmarks.loc2seq import download_dataset

project_dir = Path("/home/gavin/Projects/drexel_classes/NLP/project")

/home/gavin/.local/lib/python3.14/site-packages/genomic_benchmarks/utils/datasets.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


# 1. Download data

In [ ]:
dataset_path = Path(download_dataset(
    "human_nontata_promoters",
    version=0,
    dest_path=project_dir / "data" / "genomic_benchmarks"
))

records = []
for split in ["train", "test"]:
    for label_name, label_value in [("negative", 0), ("positive", 1)]:
        for fp in sorted((dataset_path / split / label_name).iterdir()):
            if fp.is_file():
                seq = "".join(fp.read_text().strip().upper().split())
                records.append({
                    "sequence_id": f"{split}_{label_name}_{fp.stem}",
                    "sequence": seq,
                    "label": label_value,
                    "label_name": label_name,
                    "split": split,
                })

df = pd.DataFrame(records)
out_dir = project_dir / "data/processed"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "human_nontata_promoters.csv", index=False)
print(f"{len(df)} sequences saved")

36131 sequences saved


In [ ]:
# Convert to Fasta files
fasta_dir = project_dir / "data/motif_analysis"
fasta_dir.mkdir(parents=True, exist_ok=True)

def write_fasta(sub_df, out_path):
    with open(out_path, "w") as f:
        for _, row in sub_df.iterrows():
            f.write(f">{row['sequence_id']}|label={row['label']}|split={row['split']}\n{row['sequence']}\n")

write_fasta(df[(df["split"] == "train") & (df["label"] == 1)], fasta_dir / "promoters_positive_train.fa")
write_fasta(df[(df["split"] == "train") & (df["label"] == 0)], fasta_dir / "promoters_negative_train.fa")
write_fasta(df, fasta_dir / "promoters_all.fa")
print("FASTA files written")

FASTA files written


# 2. Motif selection

Before running this block, make sure to run the tools outside this to make it actually work. Note that these are all my personal paths, od whatever works for you and ignore any personal info lol

Also, several tools need to be installed for this to work - AME and FIMO. These are included in the MEME suite, which is available through bioconda: https://anaconda.org/bioconda/meme

For the tool versions, I used 5.5.9 for both.



1. JASPAR dataset - here I downloaded the JASPAR core vertebrate motifs. This is to have motif examples, which we will use to compare against our dataset to identify enriched motifs.

```bash
mkdir -p /home/gavin/Projects/drexel_classes/NLP/project/data/motifs
cd /home/gavin/Projects/drexel_classes/NLP/project/data/motifs

wget -O jaspar_core_vertebrates.meme \
https://jaspar.elixir.no/download/data/2026/CORE/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme.txt
```
check if it was downloaded right:
```bash
head -20 jaspar_core_vertebrates.meme
grep -c "^MOTIF" jaspar_core_vertebrates.meme
```
This returned 1019 total motifs:

```bash
MEME version 4

ALPHABET= ACGT

strands: + -

Background letter frequencies
A 0.25 C 0.25 G 0.25 T 0.25

MOTIF MA0004.1 Arnt
letter-probability matrix: alength= 4 w= 6 nsites= 20 E= 0
 0.200000  0.800000  0.000000  0.000000
 0.950000  0.000000  0.050000  0.000000
 0.000000  1.000000  0.000000  0.000000
 0.000000  0.000000  1.000000  0.000000
 0.000000  0.000000  0.000000  1.000000
 0.000000  0.000000  1.000000  0.000000
URL http://jaspar.elixir.no/matrix/MA0004.1

MOTIF MA0069.1 PAX6
1019
```


2. Run AME to find enriched motifs in the promoters positive and negative datasets. The idea is to find which exact motifs are enriched (prevalent) in promoter vs non-promoter sequences. AME is a bioinformatics tool that identifies short sequences (motifs) that are enriched in a dataset vs another. It may take a minute or two.

```bash
mkdir -p /home/gavin/Projects/drexel_classes/NLP/project/results/ame_promoter_enrichment_text

ame --text\
  --control /home/gavin/Projects/drexel_classes/NLP/project/data/motif_analysis/promoters_negative_train.fa \
  /home/gavin/Projects/drexel_classes/NLP/project/data/motif_analysis/promoters_positive_train.fa \
  /home/gavin/Projects/drexel_classes/NLP/project/data/motifs/jaspar_core_vertebrates.meme \
  > /home/gavin/Projects/drexel_classes/NLP/project/results/ame_promoter_enrichment_text/ame.tsv

```



In [ ]:
# print out enriched motifs so we can make some downstream decisioins on which ones to select
ame_path = Path("/home/gavin/Projects/drexel_classes/NLP/project/results/ame_promoter_enrichment_text/ame.tsv")

ame = pd.read_csv(
    ame_path,
    sep="\t",
    comment="#"
)

print("AME rows:", len(ame))
print(ame.head(40)[[
    "rank", "motif_ID", "motif_alt_ID", "consensus",
    "p-value", "adj_p-value", "E-value",
    "TP", "%TP", "FP", "%FP"
]].to_string(index=False))

AME rows: 278
 rank motif_ID motif_alt_ID      consensus  p-value  adj_p-value  E-value   TP   %TP   FP   %FP
    1 MA0516.3          SP2      GGGGCGGGG      0.0          0.0      0.0 7196 48.81  614  4.97
    2 MA2328.1        ZBED4     CCCGCYCCGC      0.0          0.0      0.0 6972 47.29  607  4.91
    3 MA0742.2        KLF12      GGGGCGGGG      0.0          0.0      0.0 6980 47.35  744  6.02
    4 MA1961.2        PATZ1    SGGGGMGGGGS      0.0          0.0      0.0 8233 55.85 1386 11.22
    5 MA2546.1       ZNF131  CCGCGCCCGCGCS      0.0          0.0      0.0 5496 37.28  220  1.78
    6 MA1513.2        KLF15       CCCCGCCC      0.0          0.0      0.0 6126 41.55  457  3.70
    7 MA0079.5          SP1      GGGGCGGGG      0.0          0.0      0.0 6964 47.24  846  6.85
    8 MA2681.1         KLF8    RGGGGCGGGGC      0.0          0.0      0.0 7717 52.35 1254 10.15
    9 MA1511.2        KLF10      GGGGCGGGG      0.0          0.0      0.0 7140 48.43 1016  8.22
   10 MA1959.2         KLF

In [ ]:
# Extract the 6 selected motifs from JASPAR MEME file
# these are selected for high enrichment as well as diversity. Many of the really high enrichment motifs are just super cg enriched
# bits. These are kinda expected. The C base in DNA has a tendency to spontaneously turn into a T through deanimation when next to a G.
# this means that Cs followed by a G are pretty rare in DNA, because they like to mutate.
# In many important sections of dna though, this mutation can be pretty bad.
# As a result, we have evolved protections in important areas. This is clear here - many of the top
# enriched motifs have CGs next to eachother (CpG islands).
selected_ids = {
    "MA0516.3",
    "MA2328.1",
    "MA2546.1",
    "MA1122.2",
    "MA0162.5",
    "MA0759.3"
}

text = (project_dir / "data/motifs/jaspar_core_vertebrates.meme").read_text().splitlines()

header_lines, motif_blocks, current_block, in_motif = [], [], [], False
for line in text:
    if line.startswith("MOTIF "):
        if current_block:
            motif_blocks.append(current_block)
        current_block, in_motif = [line], True
    elif in_motif:
        current_block.append(line)
    else:
        header_lines.append(line)
if current_block:
    motif_blocks.append(current_block)

selected_blocks = [b for b in motif_blocks if b[0].split()[1] in selected_ids]

out_path = project_dir / "data/motifs/selected_promoter_motifs.meme"
with open(out_path, "w") as f:
    f.write("\n".join(header_lines).rstrip() + "\n\n")
    for block in selected_blocks:
        f.write("\n".join(block).rstrip() + "\n\n")
print(f"Wrote {len(selected_blocks)} motifs to {out_path}")

Wrote 6 motifs to /home/gavin/Projects/drexel_classes/NLP/project/data/motifs/selected_promoter_motifs.meme


# 3. find and label per-sequence motifs using FIMO.

Run FIMO first. This compares seqs against motifs and annotates them.

```bash
mkdir -p /home/gavin/Projects/drexel_classes/NLP/project/results/fimo_selected

fimo \
  --oc /home/gavin/Projects/drexel_classes/NLP/project/results/fimo_selected \
  --thresh 1e-4 \
  /home/gavin/Projects/drexel_classes/NLP/project/data/motifs/selected_promoter_motifs.meme \
  /home/gavin/Projects/drexel_classes/NLP/project/data/motif_analysis/promoters_all.fa
```

In [ ]:
# annotate each sequence with per-motif presence flags from FIMO results
motif_name_map = {
    "MA0516.3": "SP2",
    "MA2328.1": "ZBED4",
    "MA0162.5": "EGR1",
    "MA0759.3": "ELK3",
    "MA1122.2": "TFDP1",
    "MA2546.1": "ZNF131",
}
motif_cols = list(motif_name_map.values())

df = pd.read_csv(project_dir / "data/processed/human_nontata_promoters.csv")
fimo = pd.read_csv(project_dir / "results/fimo_selected/fimo.tsv", sep="\t", comment="#")

fimo["sequence_id"] = fimo["sequence_name"].str.split("|").str[0]
fimo["motif_short"] = fimo["motif_id"].map(motif_name_map)

presence = (
    fimo.dropna(subset=["motif_short"])
        .assign(present=1)
        .drop_duplicates(["sequence_id", "motif_short"])
        .pivot(index="sequence_id", columns="motif_short", values="present")
        .fillna(0).astype(int).reset_index()
)

annot = df.merge(presence, on="sequence_id", how="left")
for col in motif_cols:
    annot[col] = annot[col].fillna(0).astype(int)

annot["num_selected_motifs"] = annot[motif_cols].sum(axis=1)
annot["motif_combo"] = annot.apply(
    lambda row: "+".join(sorted(m for m in motif_cols if row[m] == 1)) or "none", axis=1
)

annot.to_csv(project_dir / "data/processed/human_nontata_promoters_with_selected_motifs.csv", index=False)
print(f"Annotated {len(annot)} sequences")

Annotated 36131 sequences


In [ ]:
# show motif-combo counts by label
combo_counts = (
    annot.groupby(["motif_combo", "label_name"])
    .size()
    .unstack(fill_value=0)
)

combo_counts["total"] = combo_counts.sum(axis=1)
combo_counts["pos_frac"] = combo_counts.get("positive", 0) / combo_counts["total"]

combo_counts = combo_counts.sort_values("total", ascending=False)

print("Top motif combos overall:")
display(combo_counts.head(50))

Top motif combos overall:


label_name,negative,positive,total,pos_frac
motif_combo,,,,
none,11497,3885,15382,0.252568
EGR1+SP2+TFDP1+ZBED4+ZNF131,67,3025,3092,0.978331
SP2,1530,1077,2607,0.413119
EGR1+SP2+ZBED4+ZNF131,134,2191,2325,0.942366
EGR1,742,372,1114,0.333932
SP2+ZBED4+ZNF131,114,727,841,0.864447
SP2+TFDP1+ZBED4+ZNF131,64,768,832,0.923077
ZBED4,354,251,605,0.414876
SP2+ZBED4,164,396,560,0.707143


# 4. Build dataset splits

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# we hold these out in training
heldout_combos = {
    "EGR1+SP2",
    "SP2+TFDP1",
    "SP2+ZNF131",
    "ZBED4+ZNF131",
}

# we filter everything by these motifs + motif combos.
# these have sufficient support in both negative and positive sequences in our DB
eligible_combos = {
    "none",
    "SP2",
    "EGR1",
    "ZBED4",
    "ZNF131",
    "ELK3",
    "TFDP1",
    "SP2+ZBED4",
    "EGR1+SP2",
    "SP2+TFDP1",
    "SP2+ZNF131",
    "ZBED4+ZNF131",
    "SP2+ZBED4+ZNF131",
    "EGR1+SP2+ZBED4+ZNF131",
}

df = annot[annot["motif_combo"].isin(eligible_combos)].copy()

heldout_test = df[df["motif_combo"].isin(heldout_combos)].copy()
nonheldout = df[~df["motif_combo"].isin(heldout_combos)].copy()

train_idx, iid_idx = train_test_split(
    nonheldout.index,
    test_size=0.20,
    random_state=42,
    stratify=nonheldout["label"]
)

composition_train = nonheldout.loc[train_idx].copy()
iid_test = nonheldout.loc[iid_idx].copy()

matched_parts = []

for (label, n_motifs), group in heldout_test.groupby(["label", "num_selected_motifs"]):
    pool = iid_test[
        (iid_test["label"] == label) &
        (iid_test["num_selected_motifs"] == n_motifs)
    ]

    matched_parts.append(
        pool.sample(n=len(group),replace=len(pool) < len(group),random_state=42)
    )

matched_iid_test = pd.concat(matched_parts).sample(frac=1, random_state=42)

cols = [
    "sequence_id",
    "sequence",
    "label",
    "label_name",
    "motif_combo",
    "num_selected_motifs",
    "SP2",
    "ZBED4",
    "EGR1",
    "ELK3",
    "TFDP1",
    "ZNF131",
]

out_dir = project_dir / "data/model_ready"
out_dir.mkdir(parents=True, exist_ok=True)

composition_train[cols].to_csv(out_dir / "composition_train.csv", index=False)
iid_test[cols].to_csv(out_dir / "iid_test.csv", index=False)
matched_iid_test[cols].to_csv(out_dir / "matched_iid_test.csv", index=False)
heldout_test[cols].to_csv(out_dir / "heldout_combo_test.csv", index=False)

print("composition_train:", composition_train.shape)
print("iid_test:", iid_test.shape)
print("matched_iid_test:", matched_iid_test.shape)
print("heldout_test:", heldout_test.shape)

composition_train: (19616, 13)
iid_test: (4905, 13)
matched_iid_test: (1569, 13)
heldout_test: (1569, 13)


# quick knn eval

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

data_dir = Path("/home/gavin/Projects/drexel_classes/NLP/project/data/model_ready")

train = pd.read_csv(data_dir / "composition_train.csv")

def check_acgt(df, name):
    bad = df[~df["sequence"].str.fullmatch("[ACGT]+")]
    print(name, "bad seqs:", len(bad))
    if len(bad):
        print(bad[["sequence_id", "sequence"]].head())
check_acgt(train, "train")



tests = {
    "ID": "iid_test.csv",
    "ID_2": "matched_iid_test.csv",
    "OOD": "heldout_combo_test.csv"
}

model = make_pipeline(
    TfidfVectorizer(analyzer="char", ngram_range=(4, 4), lowercase=False),
    KNeighborsClassifier(n_neighbors=5, weights="distance", metric="cosine", algorithm="brute")
)

model.fit(train["sequence"], train["label"])

rows = []
for name, file in tqdm(tests.items()):
    test = pd.read_csv(data_dir / file)
    y = test["label"]
    pred = model.predict(test["sequence"])
    prob = model.predict_proba(test["sequence"])[:, 1]

    rows.append({
        "test": name,
        "n": len(test),
        "pos_rate": y.mean(),
        "acc": accuracy_score(y, pred),
        "bal_acc": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "auroc": roc_auc_score(y, prob),
    })

pd.DataFrame(rows)

train bad seqs: 0


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:35<00:00, 11.88s/it]


,test,n,pos_rate,acc,bal_acc,macro_f1,auroc
0,ID,4905,0.387564,0.841182,0.795977,0.814375,0.938316
1,ID_2,1569,0.613767,0.859783,0.885774,0.858706,0.996525
2,OOD,1569,0.613767,0.790950,0.812876,0.789296,0.893139
